# 03. 임베딩 — embed_text 를 bge-m3 벡터로

**무엇을 하나:** 각 레시피의 `embed_text`(검색용 한 문장)를 bge-m3 로 **1024차원 벡터**로 만든다. → `data/lab/03_vecs.pkl`

**벡터에 들어가는 것 = embed_text (꼭 이해):**
```
{요리명} 재료: {재료 이름들} 조리: {조리과정} 영양: {kcal}kcal 단백질 {g}g 시간: {분}분 장르: {장르}
```
- 들어감: 요리명 · 재료 **이름** · 조리과정 · 칼로리 · 단백질 · 시간 · 장르
- **안 들어감**: 재료 **분량/단위**, 탄수, 지방 → 이건 메타데이터로만 저장(04). 분량은 검색이 아니라 장보기/표시용이라 벡터에 안 넣는다.

**왜 이렇게 하나:** 벡터는 "의미 좌표"다. 비슷한 뜻의 문장은 좌표에서 가깝다. 그래서 `"두부 매콤 한식"` 쿼리가 그 단어들이 든 레시피 embed_text 와 가까워져 검색된다. (단, "단백질 20g 이상" 같은 **숫자 조건은 벡터가 못 함** → 05~06에서 메타 필터로 처리.)

**파이프라인 대응:** `s3_embed_text.py`. (운영 s3 는 정제 결과로 embed_text 를 재생성; 노트북은 02 결과의 embed_text 를 그대로 임베딩.)

In [ ]:
import sys,os,json,asyncio
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
from app.rag.embedder import Embedder
recs=json.loads((B/'data'/'lab'/'02_norm.json').read_text(encoding='utf-8'))
if not recs: raise SystemExit('02_norm.json 비어있음 — 01,02 먼저 실행')
emb=Embedder(); print('embedder:',emb.provider,'sig:',emb.signature)
texts=[r['embed_text'] for r in recs]
vecs=[]
for i in range(0,len(texts),64): vecs.extend(asyncio.run(emb.embed_documents(texts[i:i+64])))
print('vectors:',len(vecs),'dim:',len(vecs[0]) if vecs else 0)
import pickle; (B/'data'/'lab'/'03_vecs.pkl').write_bytes(pickle.dumps({'recs':recs,'vecs':vecs}))
print('saved -> data/lab/03_vecs.pkl')